## Installations

#### Comments:
SI of paper states "all models were trained using Chemprop version 1.5" - version 1.5.2 installed as this was the most recent 1.5 version at the time the paper was published

Older version of scikit-learn, numpy, and pandas also installed to match time period of paper, and because errors were obtained later on due to changes made between then and now

Removed these as instead of uninstalling then reinstalling specific package/Python versions I did so in Anaconda Prompt and saved the environment as a kernel ("deep4chem")

## Imports

In [53]:
import sys
import pandas as pd
import numpy as np
import os
import chemprop
import sklearn

In [4]:
print(sklearn.__version__)
print(chemprop.__version__)
print(np.__version__)
print(sys.version)

1.2.2
1.5.2
1.24.4
3.10.20 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:13:20) [MSC v.1942 64 bit (AMD64)]


## Loading and Viewing the Datasets

In [90]:
# creating a variable to store the absolute .csv file path for the full Deep4Chem dataset
d4c_path = r'C:/Users/Amy/Desktop/Masters_Project/paper_methods/Deep4Chem_chemprop/Deep4Chem_chemprop/d4c_ext_coef_all_data.csv'

# reading in the dataset as stored in the path variable
d4c = pd.read_csv(d4c_path)

# printing the index and heading of each column in the dataset
for i, col in enumerate(d4c.columns):
    print(i, repr(col))

# viewing the shape (no. entries) and title of columns contained by the data
print('Shape:', d4c.shape)
print('\nColumns:')
for col in d4c.columns:
    print(repr(col))

# checking for missing values within the dataset
print('\nMissing values:')
print(d4c.isnull().sum())

0 'Chromophore'
1 'Solvent'
2 'Absorption max (nm)'
3 'log(e/mol-1 dm3 cm-1)'
Shape: (8032, 4)

Columns:
'Chromophore'
'Solvent'
'Absorption max (nm)'
'log(e/mol-1 dm3 cm-1)'

Missing values:
Chromophore              0
Solvent                  0
Absorption max (nm)      0
log(e/mol-1 dm3 cm-1)    0
dtype: int64


In [91]:
# creating a variable to store the absolute .csv file path for the full Reaxys dataset
reaxys_path = r'C:/Users/Amy/Desktop/Masters_Project/paper_methods/reaxys_dataset/reaxys_data.csv'

# reading in the dataset as stored in the path variable
reaxys = pd.read_csv(reaxys_path)

# printing the index and heading of each column in the dataset
for i, col in enumerate(reaxys.columns):
    print(i, repr(col))

# viewing the shape (no. entries) and title of columns contained by the data
print('Shape:', reaxys.shape)
print('\nColumns:')
for col in reaxys.columns:
    print(repr(col))

# checking for missing values within the dataset
print('\nMissing values:')
print(reaxys.isnull().sum())

0 'SMILES'
1 'SOLVENT'
2 'logE'
3 'UVmax'
Shape: (33768, 4)

Columns:
'SMILES'
'SOLVENT'
'logE'
'UVmax'

Missing values:
SMILES     0
SOLVENT    0
logE       0
UVmax      0
dtype: int64


#### Comments:
8,032 datapoints in Deep4Chem dataset instead of ~3.8k, 33,768 in Reaxys unlike ~38k as reported

No missing values in either => handy for handling datasets later

Column names slightly different and in different orders - will fix so following scripts will be generalisable

## Initial Training Command (Replication of Deep4Chem_chemprop Method)

In [6]:
# creating a command from important arguments/parameters within the 'verbose.log' and 'args.json' files attached by the authors
cmd = (
    f'chemprop_train '
    f'--data_path "{csv_path}" '
    f'--dataset_type regression '
    f'--num_folds 10 '
    f'--epochs 10 '
    f'--number_of_molecules 2 '
    f'--split_type random '
    f'--seed 0 '
    f'--depth 5 '
    f'--hidden_size 1900 '
    f'--ffn_hidden_size 1900 '
    f'--ffn_num_layers 3 '
    f'--dropout 0.1 '
    f'--aggregation mean '
    f'--batch_size 50 '
    f'--metric rmse '
    f'--save_dir d4c_replication'
)

print(cmd)

chemprop_train --data_path "C:/Users/Amy/Desktop/Masters_Project/paper_methods/Deep4Chem_chemprop/Deep4Chem_chemprop/d4c_ext_coef_all_data.csv" --dataset_type regression --num_folds 10 --epochs 10 --number_of_molecules 2 --split_type random --seed 0 --depth 5 --hidden_size 1900 --ffn_hidden_size 1900 --ffn_num_layers 3 --dropout 0.1 --aggregation mean --batch_size 50 --metric rmse --save_dir d4c_replication


In [7]:
# executing training command
os.system(cmd)

0

#### Comments:

Reduced the number of epochs from 200 to 10 due to time limitations running the command on CPU after ~32 epochs

Output of model contained within the 'd4c_replication' directory (later renamed to 'Deep4Chem_chemprop_REPLICATION' to more obviously indicate what part of the repository/work's method was being replicated) - contains each fold's model and test scores, as well as logs (quiet and verbose) and a CSV file of the final test scores (average and individual wavelength and log(molar extinction coefficients) for all folds, plus their standard deviations (SDs))

Output implies training was insufficient and each model was equally underfitting the data (higher mean RMSE values, lower SDs for each compared to the authors' results) => may repeat later with e.g. 100 epochs and no cross-validation (or less folds) to train more effectively within a reasonable wall time utilising the CPU